# Result 2 — Efficiency Divergence

This notebook compares realized efficiency across conditions.

Metrics:
- **Realized welfare** = sum of final utilities across players.
- **Optimal welfare** = maximum possible sum of utilities over all feasible allocations of the fixed total inventory.
- **Efficiency ratio** = realized welfare / optimal welfare.
- **Inequality** = Gini coefficient over final utilities and utility gains.

Optional sanity baseline:
- Random no-intelligence baseline generated by random feasible allocations of the same total goods.


In [2]:

from __future__ import annotations

import json
import math
import itertools
import random
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Tuple

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import yaml
except ImportError as exc:
    raise ImportError("Please install PyYAML: pip install pyyaml") from exc

OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def collect_run_dirs(parent: Optional[str | Path]) -> List[Path]:
    if parent is None:
        return []
    parent = Path(parent)
    if not parent.exists():
        raise FileNotFoundError(parent)
    return sorted([p for p in parent.iterdir() if p.is_dir() and (p / "summary.json").exists()])


def load_json(path: Path, default=None):
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_yaml(path: Path, default=None):
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def run_id(run_dir: Path) -> str:
    return Path(run_dir).name


def load_players_yaml(run_dir: Path) -> Dict[str, Any]:
    return load_yaml(run_dir / "config_snapshot" / "players.yaml")


def load_summary(run_dir: Path) -> Dict[str, Any]:
    return load_json(run_dir / "summary.json")


def load_preference_probes(run_dir: Path) -> List[Dict[str, Any]]:
    return load_json(run_dir / "preference_probes.json", default=[])


def load_trades(run_dir: Path) -> List[Dict[str, Any]]:
    return load_json(run_dir / "trades.json", default=[])


def shifted_cobb_douglas(inventory: Mapping[str, int], weights: Mapping[str, float], shift: float = 1.0) -> float:
    u = 1.0
    for good, alpha in weights.items():
        u *= (inventory.get(good, 0) + shift) ** alpha
    return u


def gini(values: Iterable[float]) -> float:
    x = np.array(list(values), dtype=float)
    if len(x) == 0:
        return np.nan
    if np.allclose(x, 0):
        return 0.0
    if np.min(x) < 0:
        x = x - np.min(x)
    x = np.sort(x)
    n = len(x)
    return float((2 * np.sum((np.arange(1, n + 1) * x))) / (n * np.sum(x)) - (n + 1) / n)


def normalize_probe_time(round_index: int) -> str:
    if round_index == 0:
        return "t=0"
    if round_index == -1:
        return "post"
    return f"t={round_index}"


def ordered_probe_time_value(label: str) -> int:
    if label == "t=0":
        return 0
    if label == "post":
        return 999999
    if label.startswith("t="):
        try:
            return int(label.split("=")[1])
        except ValueError:
            return 999998
    return 999998


In [3]:

# ---------------------------------------------------------------------
# Configure paths here
# ---------------------------------------------------------------------
CONTROL_RUN_DIRS = [
    # Path("../runs/20260603_120000_condition_1_control_silent_barter"),
]

BROADCAST_RUN_DIRS = [
    # Path("../runs/20260603_130000_condition_2_broadcast"),
]

NO_INTELLIGENCE_RUN_DIRS = [
    # Optional: explicit no-intelligence baseline run folders.
]

CONTROL_PARENT = "/home/yyspencer/PSU/REAL/Preference_drift_results/Runs_6_3/silent barter runs"     # e.g., "../runs/control"
BROADCAST_PARENT = "/home/yyspencer/PSU/REAL/Preference_drift_results/Runs_6_3/broadcast runs"    # e.g., "../runs/broadcast"
NO_INTELLIGENCE_PARENT = "/home/yyspencer/PSU/REAL/Preference_drift_results/random runs"

control_dirs = list(CONTROL_RUN_DIRS) + collect_run_dirs(CONTROL_PARENT)
broadcast_dirs = list(BROADCAST_RUN_DIRS) + collect_run_dirs(BROADCAST_PARENT)
baseline_dirs = list(NO_INTELLIGENCE_RUN_DIRS) + collect_run_dirs(NO_INTELLIGENCE_PARENT)

print(f"Control runs: {len(control_dirs)}")
print(f"Broadcast runs: {len(broadcast_dirs)}")
print(f"No-intelligence runs: {len(baseline_dirs)}")


Control runs: 8
Broadcast runs: 8
No-intelligence runs: 8


In [4]:

def get_players_from_snapshot(run_dir: Path) -> List[Dict[str, Any]]:
    players_yaml = load_players_yaml(run_dir)
    return players_yaml.get("players", [])


def total_initial_goods(players: List[Dict[str, Any]]) -> Dict[str, int]:
    goods = sorted(players[0]["inventory"].keys())
    totals = {g: 0 for g in goods}
    for p in players:
        for g, q in p["inventory"].items():
            totals[g] += int(q)
    return totals


def generate_allocations_for_one_good(total: int, n_players: int):
    """All ways to distribute `total` identical units across n_players."""
    if n_players == 1:
        yield (total,)
        return
    for x in range(total + 1):
        for rest in generate_allocations_for_one_good(total - x, n_players - 1):
            yield (x,) + rest


def brute_force_optimal_welfare(players: List[Dict[str, Any]], shift: float = 1.0) -> Tuple[float, List[Dict[str, int]]]:
    """
    Maximize sum_i U_i(x_i) subject to fixed total units for each good.
    Works well for small pilots, e.g. 6 players, 3 goods.
    """
    goods = sorted(players[0]["inventory"].keys())
    n = len(players)
    totals = total_initial_goods(players)

    per_good_allocs = [list(generate_allocations_for_one_good(totals[g], n)) for g in goods]

    best_welfare = -float("inf")
    best_allocation = None

    for combo in itertools.product(*per_good_allocs):
        inventories = []
        for i in range(n):
            inv = {g: combo[g_idx][i] for g_idx, g in enumerate(goods)}
            inventories.append(inv)

        welfare = 0.0
        for p, inv in zip(players, inventories):
            welfare += shifted_cobb_douglas(inv, p["utility_weights"], shift=shift)

        if welfare > best_welfare:
            best_welfare = welfare
            best_allocation = inventories

    return best_welfare, best_allocation


def random_feasible_allocation(totals: Dict[str, int], n_players: int, rng: random.Random) -> List[Dict[str, int]]:
    goods = sorted(totals.keys())
    inventories = [{g: 0 for g in goods} for _ in range(n_players)]
    for g in goods:
        for _ in range(totals[g]):
            inventories[rng.randrange(n_players)][g] += 1
    return inventories


def random_baseline_welfare(players: List[Dict[str, Any]], n_samples: int = 5000, seed: int = 123, shift: float = 1.0) -> Dict[str, float]:
    rng = random.Random(seed)
    totals = total_initial_goods(players)
    welfare_values = []
    for _ in range(n_samples):
        allocation = random_feasible_allocation(totals, len(players), rng)
        welfare = sum(
            shifted_cobb_douglas(inv, p["utility_weights"], shift=shift)
            for p, inv in zip(players, allocation)
        )
        welfare_values.append(welfare)
    arr = np.array(welfare_values, dtype=float)
    return {
        "random_baseline_mean_welfare": float(arr.mean()),
        "random_baseline_sd_welfare": float(arr.std(ddof=1)),
        "random_baseline_p05_welfare": float(np.percentile(arr, 5)),
        "random_baseline_p95_welfare": float(np.percentile(arr, 95)),
    }


In [15]:

def extract_efficiency_for_run(
    run_dir: Path,
    condition: str,
    compute_optimal: bool = True,
    compute_random_baseline: bool = True,
    random_samples: int = 5000,
) -> Dict[str, Any]:
    summary = load_summary(run_dir)
    players = get_players_from_snapshot(run_dir)
    shift = 1.0

    starting_utils = summary.get("starting_utilities", [])
    final_utils = summary.get("final_utilities", [])

    realized_welfare = sum(float(x["final_utility"]) for x in final_utils)
    starting_welfare = sum(float(x["utility"]) for x in starting_utils)

    utility_by_player = {x["player_id"]: float(x["final_utility"]) for x in final_utils}
    start_by_player = {x["player_id"]: float(x["utility"]) for x in starting_utils}
    gains = [utility_by_player[pid] - start_by_player.get(pid, 0.0) for pid in utility_by_player.keys()]

    row = {
        "condition": condition,
        "run_id": run_id(run_dir),
        "run_dir": str(run_dir),
        "num_players": summary.get("num_players"),
        "num_rounds": summary.get("num_rounds"),
        "goods": ",".join(summary.get("goods", [])),
        "total_trades_accepted": summary.get("total_trades_accepted"),
        "total_trades_rejected": summary.get("total_trades_rejected"),
        "starting_welfare": starting_welfare,
        "realized_welfare": realized_welfare,
        "welfare_change": realized_welfare - starting_welfare,
        "gini_final_utility": gini(utility_by_player.values()),
        "gini_utility_gain": gini(gains),
    }

    if compute_optimal:
        optimal_welfare, optimal_allocation = brute_force_optimal_welfare(players, shift=shift)
        row["optimal_welfare"] = optimal_welfare
        row["efficiency_ratio"] = realized_welfare / optimal_welfare if optimal_welfare else np.nan
        row["optimal_allocation"] = json.dumps(optimal_allocation)

    if compute_random_baseline:
        baseline = random_baseline_welfare(players, n_samples=random_samples, seed=123, shift=shift)
        row.update(baseline)
        if "optimal_welfare" in row:
            row["random_baseline_efficiency_mean"] = row["random_baseline_mean_welfare"] / row["optimal_welfare"]

    return row


rows = []
for run_dir in control_dirs:
    rows.append(extract_efficiency_for_run(Path(run_dir), "Control"))
for run_dir in broadcast_dirs:
    rows.append(extract_efficiency_for_run(Path(run_dir), "Broadcast"))
for run_dir in baseline_dirs:
    rows.append(extract_efficiency_for_run(Path(run_dir), "No-intelligence"))

eff_df = pd.DataFrame(rows)
if not eff_df.empty:
    eff_df.to_csv(OUTPUT_DIR / "result2_efficiency_raw.csv", index=False)
eff_df


Total supply: {'A': 8, 'B': 6, 'C': 10}
Per-good allocation counts: {'A': 1287, 'B': 462, 'C': 3003}
Total brute-force combinations: 1785565782


RuntimeError: Brute-force search too large: 1,785,565,782 combinations. Set compute_optimal=False or use approximate_optimal_welfare instead.

In [ ]:

def summarize_efficiency(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    metrics = [
        "efficiency_ratio",
        "realized_welfare",
        "welfare_change",
        "gini_final_utility",
        "gini_utility_gain",
        "total_trades_accepted",
    ]
    metrics = [m for m in metrics if m in df.columns]

    grouped = df.groupby("condition").agg(
        **{f"{m}_mean": (m, "mean") for m in metrics},
        **{f"{m}_sd": (m, "std") for m in metrics},
        n=("run_id", "count"),
    ).reset_index()

    for m in metrics:
        grouped[f"{m}_se"] = grouped[f"{m}_sd"] / np.sqrt(grouped["n"])

    return grouped


summary2 = summarize_efficiency(eff_df)
if not summary2.empty:
    summary2.to_csv(OUTPUT_DIR / "result2_efficiency_summary.csv", index=False)
summary2


In [ ]:

def barplot_metric(summary: pd.DataFrame, metric: str, ylabel: str, title: str, out_path: Path):
    if summary.empty:
        print("No data to plot.")
        return

    mean_col = f"{metric}_mean"
    se_col = f"{metric}_se"
    plot_df = summary.dropna(subset=[mean_col]).copy()
    x = np.arange(len(plot_df))

    plt.figure(figsize=(7, 5))
    plt.bar(x, plot_df[mean_col].to_numpy())

    if se_col in plot_df.columns:
        plt.errorbar(x, plot_df[mean_col].to_numpy(), yerr=plot_df[se_col].fillna(0).to_numpy(), fmt="none", capsize=4)

    plt.xticks(x, plot_df["condition"].tolist())
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()


barplot_metric(
    summary2,
    metric="efficiency_ratio",
    ylabel="Efficiency ratio: realized welfare / optimal welfare",
    title="Efficiency Divergence Across Conditions",
    out_path=OUTPUT_DIR / "result2_efficiency_ratio.png",
)


In [ ]:

barplot_metric(
    summary2,
    metric="gini_final_utility",
    ylabel="Gini coefficient over final utilities",
    title="Distributional Inequality Across Conditions",
    out_path=OUTPUT_DIR / "result2_final_utility_gini.png",
)


In [ ]:

# Optional sanity-check plot: realized efficiency vs random no-intelligence baseline.
if not eff_df.empty and "random_baseline_efficiency_mean" in eff_df.columns:
    baseline_summary = eff_df.groupby("condition", as_index=False).agg(
        realized_efficiency=("efficiency_ratio", "mean"),
        random_baseline_efficiency=("random_baseline_efficiency_mean", "mean"),
    )
    baseline_summary.to_csv(OUTPUT_DIR / "result2_random_baseline_comparison.csv", index=False)
    display(baseline_summary)

    x = np.arange(len(baseline_summary))
    width = 0.35

    plt.figure(figsize=(8, 5))
    plt.bar(x - width / 2, baseline_summary["realized_efficiency"], width, label="Realized")
    plt.bar(x + width / 2, baseline_summary["random_baseline_efficiency"], width, label="Random baseline")
    plt.xticks(x, baseline_summary["condition"].tolist())
    plt.ylabel("Efficiency ratio")
    plt.title("Realized Efficiency vs No-Intelligence Baseline")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "result2_realized_vs_random_baseline.png", dpi=200)
    plt.show()
